# Ch.3 — Matrix Factorization

> **The story.** In **1901** Karl Pearson introduced principal components — compressing datasets into fewer dimensions without losing the signal. By the **1960s**, Singular Value Decomposition (SVD) was formalised: any matrix $R = U \Sigma V^T$. The trouble: SVD requires knowing the _entire_ matrix, but a recommender system's ratings matrix is 93.7% empty. The breakthrough came in **2006** when Simon Funk, competing in the **Netflix Prize**, published "Netflix Update: Try This at Home" — only train on ratings you _can_ see, using SGD one rating at a time. "Funky SVD" rocketed him to third on the leaderboard overnight. **Yehuda Koren** at Yahoo! Research formalised the framework with bias terms and temporal dynamics; the BellKor team won the Netflix Prize with 100+ blended models, regularised matrix factorisation as the backbone. In **2012** Steffen **Rendle** published **BPR**, extending factorization to implicit feedback via pairwise ranking — the foundation of modern "Learn to Rank" systems.
>
> **Where you are in the curriculum.** Chapter three. Collaborative filtering (Ch.2) achieved ~65% HR@10 but is crippled by sparsity — most user pairs share fewer than 5 movies. Matrix factorization solves this by mapping users and items into a shared **latent space** where the dot product approximates a rating. Even if two users never rated the same movie, their vectors can be close — capturing "cerebral sci-fi lover" or "90s comedy fan" without those labels ever being defined. This is your first encounter with _latent representations_ — the same idea powering word embeddings, BERT, and every deep learning model in this track.
>
> **Notation.** $R \in \mathbb{R}^{m \times n}$ — rating matrix ($m$ users, $n$ items); $P \in \mathbb{R}^{m \times k}$, $Q \in \mathbb{R}^{n \times k}$ — user/item factor matrices; $k$ — latent factors; $\hat{r}_{ui} = \mathbf{p}_u^\top \mathbf{q}_i + \mu + b_u + b_i$ — predicted rating; $\lambda$ — L2 regularisation; $e_{ui} = r_{ui} - \hat{r}_{ui}$ — prediction error.

---

## §0 · The Challenge — Where We Are

> **The mission**: Launch **FlixAI** — >85% HR@10 across 5 constraints: (1) ACCURACY >85%, (2) COLD START for new users/items, (3) SCALABILITY <200ms, (4) DIVERSITY beyond blockbusters, (5) EXPLAINABILITY "Because you liked X."

**Progress so far:** Ch.1 popularity → 42% HR@10. Ch.2 item-CF → ~65% HR@10. **Still 20 points short.** Item-CF stores an $O(n^2)$ similarity matrix (2.8M floats for 1,682 movies) and fails whenever user pairs share too few ratings. MF with $k=50$ factors compresses everything into 131k floats — 21× more compact — and fills in missing entries through learned structure. The key insight: MF discovers **latent "genre-taste" dimensions** automatically — the dot product $\mathbf{p}_u^\top \mathbf{q}_i$ becomes a taste-to-attribute alignment score.

```mermaid
flowchart LR
 CF["Ch.2: Item-CF\nHR@10 ≈ 65%\nO(n²) similarity"] --> DECOMP["Learn P ∈ ℝ^(m×k)\nQ ∈ ℝ^(n×k)\nvia SGD"]
 DECOMP --> PRED["Predict:\nr̂ = pᵤᵀqᵢ"]
 PRED --> BIAS["+ Bias terms:\nμ + bᵤ + bᵢ"]
 BIAS --> EVAL["MF + bias\nHR@10 ≈ 78%"]

 style CF fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style DECOMP fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style EVAL fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Dataset:** MovieLens 100k | **Task:** SGD matrix factorization with bias terms | **Outcome:** MF = ~78% HR@10


## §1 · The Core Idea

Every user and every movie can be described by a small set of hidden ("latent") preferences. A sci-fi fan has a high score on the "cerebral sci-fi" dimension; _Blade Runner_ has a high score on that same dimension. **You never define these dimensions** — the model discovers them by finding the best $k$-dimensional vectors that reconstruct observed ratings. The dot product between a user vector and an item vector then measures how "aligned" their tastes and attributes are.

```
     R       ≈  P    ·   Qᵀ
  ─────────    ──────   ──────
  [m × n]     [m × k]  [k × n]

  R:  943 users × 1,682 items  (93.7% empty — 100k of 1.6M cells filled)
  P:  943 users   × k latent factors  (learned)
  Q:  1,682 items × k latent factors  (learned)
  k = 20  →  only (943+1682)×20 = 52,500 numbers replace 1.6M
```

The model only trains on the 6.3% of cells that have real ratings. The latent structure fills in the rest — giving a meaningful score for every user-item pair, including those that share zero ratings.

> **Optional depth:** $\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$ — global mean $\mu$ plus user bias $b_u$ plus item bias $b_i$ plus latent alignment $\mathbf{p}_u^\top \mathbf{q}_i$.

**Training objective** — minimise reconstruction error on observed ratings, penalise large vectors (L2 regularisation prevents overfitting):

> **Optional depth:** $\mathcal{L} = \sum_{(u,i) \in \text{obs}} (r_{ui} - \hat{r}_{ui})^2 + \lambda(\|\mathbf{p}_u\|^2 + \|\mathbf{q}_i\|^2 + b_u^2 + b_i^2)$


```mermaid
flowchart LR
    USER["User u\n(943 users)"] --> PVEC["p_u ∈ ℝᵏ\nUser latent vector\n(learned by SGD)"]
    ITEM["Item i\n(1,682 items)"] --> QVEC["q_i ∈ ℝᵏ\nItem latent vector\n(learned by SGD)"]
    PVEC --> DOT["Dot product\np_u · q_i\n+ μ + b_u + b_i"]
    QVEC --> DOT
    DOT --> PRED["r̂_ui\nPredicted rating\n(any user-item pair)"]

    style USER fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style ITEM fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style PVEC fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style QVEC fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style DOT fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style PRED fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

_Visual takeaway: every user-item score is a single dot product — $O(k)$ regardless of how sparse the ratings matrix is._


In [ ]:
# TODO: Implement this cell
#  (Imports)
#
# Steps:
# 1. Imports
# 2. Compute `SEED` using `set_theme()`
# 3. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Load MovieLens 100k & Split)
#
# Steps:
# 1. Load MovieLens 100k & Split
# 2. Compute `ratings` using `read_csv()`
# 3. Compute `n_users`
# 4. Aggregate data into `ratings_sorted` -- use `copy()`
# 5. Process data
#
# Hint:
#    ratings = pd.read_csv(???)
#    ratings_sorted = ratings.sort_values(???)
#    test = ratings_sorted.groupby(???)
#    train = ratings_sorted.drop(???)

In [ ]:
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #3: Implement `hit_rate_at_k()`.

    Steps:
    1. Evaluation Metrics
    2. Define helper function `ndcg_at_k()`
    3. Process data

    Hint:
    recs = top_k_per_user.get(???)
    rank = recs.index(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement hit_rate_at_k()")


def ndcg_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #3: Implement `ndcg_at_k()`.

    Steps:
    1. Evaluation Metrics
    2. Define helper function `ndcg_at_k()`
    3. Process data

    Hint:
    recs = top_k_per_user.get(???)
    rank = recs.index(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement ndcg_at_k()")

## §2 · SGD-Based Matrix Factorization

Collaborative filtering computed similarity once and looked up neighbours at query time. Matrix factorization learns _continuously_ — each rating in the training set is a teaching signal that nudges the user and item vectors closer together (or further apart, for bad predictions). The SGD update is a hill-climb in vector space: find the prediction error, then adjust each vector in the direction that reduces it, while pulling vectors toward zero (regularisation) to prevent overfitting on the sparse 6.3% of known ratings.

For each observed rating $(u, i, r_{ui})$:

1. Predict: $\hat{r} = \mu + b_u + b_i + \mathbf{p}_u^T \mathbf{q}_i$
2. Error: $e = r_{ui} - \hat{r}$
3. Update:
   - $\mathbf{p}_u \leftarrow \mathbf{p}_u + \eta(e \cdot \mathbf{q}_i - \lambda \cdot \mathbf{p}_u)$
   - $\mathbf{q}_i \leftarrow \mathbf{q}_i + \eta(e \cdot \mathbf{p}_u - \lambda \cdot \mathbf{q}_i)$
   - $b_u \leftarrow b_u + \eta(e - \lambda \cdot b_u)$
   - $b_i \leftarrow b_i + \eta(e - \lambda \cdot b_i)$

> **Optional depth:** The update rule is gradient descent on $\mathcal{L}$. $\partial\mathcal{L}/\partial\mathbf{p}_u = -2e\mathbf{q}_i + 2\lambda\mathbf{p}_u$, flipped gives the ascent direction above. $\eta = 0.005$, $\lambda = 0.02$ are typical starting values.


In [ ]:
# TODO: Implement this cell
#  (Matrix Factorization (from scratch))
#
# Steps:
# 1. Matrix Factorization (from scratch)
# 2. Call `normal()` to produce the result
# 3. Define helper function
# 4. Call `astype()` to produce the result
# 5. Call `permutation()` to produce the result
# 6. Process data
# 7. Call `sqrt()` to produce the result
# 8. Process data
# 9. Define helper function
# 10. Process data
#
# Hint:
#    U = np.random.normal(???)
#    V = np.random.normal(???)
#    b_u = np.zeros(???)
#    b_i = np.zeros(???)

In [ ]:
# TODO: Implement this cell
#  (Train the Model)
#
# Steps:
# 1. Train the Model
#
# Hint:
#    mf = MatrixFactorization(n_users=???, n_items=???)
#    losses = mf.fit(???)
#    mf.fit(???)

In [ ]:
# TODO: Implement this cell
#  (Training Loss Curve)
#
# Steps:
# 1. Training Loss Curve
#
# Hint:
#    ax = plt.subplots(???)

In [ ]:
# TODO: Implement this cell
#  (Evaluate)
#
# Steps:
# 1. Evaluate
# 2. Compute `top_k_mf` using `unique()`
# 3. Compute `hr_mf` using `hit_rate_at_k()`
# 4. Call `Factorization()` to produce the result
#
# Hint:
#    user_rated_train = train.groupby(???)
#    rated = user_rated_train.get(???)
#    recs = mf.recommend(???)

### What §1–§2 established — and what it still doesn't solve

[Done] **Sparsity solved.** The latent space provides a similarity score for every user-item pair — even those that share zero rated movies.

[Done] **Scalability improved.** Prediction is $O(k)$ — a single dot product.

[Done] **HR@10 lifted.** From ~65% (Ch.2) to ~78% — a 13-point gain from learned structure alone.

**Still open:**

- **Cold start:** A new user has no vector in $P$. The model cannot recommend anything without at least one rating.
- **Linearity ceiling:** The dot product is fundamentally linear — cannot encode "likes A and B separately but hates A+B together." Ch.4 breaks this ceiling.
- **Explainability:** Latent dimensions are discovered, not labelled.


In [ ]:
# TODO: Implement this cell
#  (Effect of Latent Factors (d))
#
# Steps:
# 1. Effect of Latent Factors (d)
# 2. Fit the model -- call `MatrixFactorization()`
# 3. Call `unique()` to produce the result
# 4. Plot results -- call `subplots()`
# 5. Process data
#
# Hint:
#    mf_d = MatrixFactorization(n_factors=???, lr=???)
#    rated = user_rated_train.get(???)
#    ax = plt.subplots(???)
#    mf_d.fit(???)

In [ ]:
# TODO: Implement this cell
#  (Visualise Latent Factors (2D PCA))
#
# Steps:
# 1. Visualise Latent Factors (2D PCA)
# 2. Fit the model -- call `PCA()`
# 3. Compute `movies_df` using `read_csv()`
# 4. Compute `genre_cols` using `idxmax()`
# 5. Plot results -- call `subplots()`
# 6. Plot results -- call `Projection()`
# 7. Process data
#
# Hint:
#    pca = PCA(n_components=???, random_state=???)
#    V_2d = pca.fit_transform(???)
#    movies_df = pd.read_csv(???)
#    ax = plt.subplots(???)

In [ ]:
# TODO: Implement this cell
#  (Regularisation Effect)
#
# Steps:
# 1. Regularisation Effect
# 2. Fit the model -- call `MatrixFactorization()`
# 3. Call `unique()` to produce the result
# 4. Plot results -- call `subplots()`
# 5. Process data
#
# Hint:
#    mf_r = MatrixFactorization(n_factors=???, lr=???)
#    rated = user_rated_train.get(???)
#    ax = plt.subplots(???)
#    mf_r.fit(???)

## Progress Check

**Checkpoint:** FlixAI — hit@10 advanced from ~65% toward the >85% target in this chapter. Matrix factorization's latent representations solve the sparsity problem that blocked collaborative filtering.

| #   | Constraint     | Target                | Ch.3 Status                                 |
| --- | -------------- | --------------------- | ------------------------------------------- |
| 1   | ACCURACY       | >85% HR@10            | ~78% (+13 pts from CF)                      |
| 2   | COLD START     | New users/items       | [No] New user = no trained vector           |
| 3   | SCALABILITY    | 1M+ ratings           | SGD scales well — $O(k)$ per step           |
| 4   | DIVERSITY      | Not just popular      | Latent space surfaces niche items           |
| 5   | EXPLAINABILITY | "Because you liked X" | Latent factors are discovered, not labelled |

**Bottom line**: 78% hit rate — latent factors handle sparsity. But the dot product is linear and cannot capture complex taste interactions.

**Next**: Ch.4 — Neural Collaborative Filtering → replace the dot product with a neural network.


## Exercises

**Exercise 1 — Bias Ablation**
Train MF without bias terms ($b_u = b_i = 0$). Compare RMSE and HR@10 against the full model with biases. How much do biases contribute?

**Exercise 2 — ALS Implementation**
Implement Alternating Least Squares: fix V and solve for each $\mathbf{u}_u$ in closed form, then fix U and solve for each $\mathbf{v}_i$. Compare convergence speed against SGD.

**Exercise 3 — Surprise Library**
Use the `surprise` library's SVD implementation. Compare HR@10 against your from-scratch MF. Are there differences?


In [ ]:
# TODO: Implement this cell
#  (Exercise 1 scaffold — Bias Ablation)
#
# Steps:
# 1. Set up: Exercise 1 scaffold — Bias Ablation
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Exercise 2 scaffold — ALS)
#
# Steps:
# 1. Set up: Exercise 2 scaffold — ALS
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Exercise 3 scaffold — Surprise Library)
#
# Steps:
# 1. Set up: Exercise 3 scaffold — Surprise Library
# 2. Process data
#
# Hint:
#    reader = Reader(rating_scale=???)
#    algo = SVD(n_factors=???, lr_all=???)
#    data = Dataset.load_from_df(???)